In [ ]:
from operator import sub

import numpy as np
from matplotlib import pyplot as plt
from nd2reader import ND2Reader

# **Step 1:** Read images of different channels

We want a dictionary of images of the different channels, like so:

```
images = {
    'channel_1_name': image_channel_1,
    'channel_2_name': image_channel_2,
    ...
}
```

Additionally, we want an array of pixel sizes. The unit can be arbitrary, but should be specified:

```
pixel_size = [pixel_size_z, pixel_size_y, pixel_size_x]
pixel_unit = 'micron'
```

Also, we want to know whether we are moving towards to sample with increasing z planes or away from sample (to coverslip), so the saved transformations can also be applied to images imaged in the opposite direction. Note that ```z_direction``` can be left blank, i.e., set to ```None```, if the direction is unknown - then it can not be considered when applying the transformation though.

```
z_direction = 'to_sample' | 'from_sample' | None
```

**We support several input options, please only run one of the blocks below to read data!**

## **Option 1:** Nikon nd2 files

In [ ]:
image_path = 'C:/Users/david/Downloads/20230414-tetraspeck-spinningdisk/20230414_tetraspeck_coverslip_001.nd2'

# spinning disk data may have additional magnification of 1.5x
# leave at 1.0 unless you are sure you used the extra zoom
magnification = 1.0

# read all channels into dict of channel_name -> img
images = {}
with ND2Reader(image_path) as reader:

    reader.bundle_axes = ['z', 'y', 'x']
    reader.iter_axes = ['c']

    for i, channel_name in enumerate(reader.metadata['channels']):
        images[channel_name.strip()] = np.array(reader[i])

    psz_xy = reader.metadata['pixel_microns'] / magnification
    # difference of z position of first two planes -> z-spacing
    psz_z = sub(*reader.metadata['z_coordinates'][:2])
    z_direction = 'to_sample' if psz_z > 0 else 'from_sample'
    # for pixel size, use absolute spacing
    psz_z = abs(psz_z)

# pixel size to array
pixel_size = np.array([psz_z, psz_xy, psz_xy])
pixel_unit = 'micron'

print(f'read nd2 file with {len(images)} channels: {list(images.keys())}')
print('image shapes:')
for channel_name, v in images.items():
    print(f'{channel_name}: {v.shape}')
print(f'pixel size: {pixel_size} {pixel_unit}')

## **Option 2:** OMX dv files (Nup153 alignment samples)

In [ ]:
from mrc import DVFile

image_path = '/Users/david/Downloads/20220408_Nup_002_SIR.dv'

with DVFile(image_path) as f:
    img = f.asarray()
    images = {str(f.hdr.__getattribute__(f'wave{i+1}')): img[i].squeeze() for i in range(f.hdr.nc)}
    pixel_size = np.array([f.hdr.dz, f.hdr.dy, f.hdr.dx])
    pixel_unit = 'micron'

z_direction = None

print(f'read DV file with {len(images)} channels: {list(images.keys())}')
print('image shapes:')
for channel_name, v in images.items():
    print(f'{channel_name}: {v.shape}')
print(f'pixel size: {pixel_size} {pixel_unit}')


## **Option 3**: Multiple TIFF files (one per channel)

As an example, we use the classic Multiview Reconstruction *Drosophila* LSFM dataset

In [ ]:
from tifffile import imread

# load 3 angles as "channels"
images = {
    '0': imread('/Volumes/davidh-ssd/HisYFP-SPIM/spim_TL18_Angle0.lsm'),
    '45': imread('/Volumes/davidh-ssd/HisYFP-SPIM/spim_TL18_Angle45.lsm'),
    '90': imread('/Volumes/davidh-ssd/HisYFP-SPIM/spim_TL18_Angle90.lsm')
}

pixel_size = np.array([2, 0.73, 0.73])
pixel_unit = 'micron'
z_direction = None

print(f'read TIFF/LSM files with {len(images)} channels: {list(images.keys())}')
print('image shapes:')
for channel_name, v in images.items():
    print(f'{channel_name}: {v.shape}')
print(f'pixel size: {pixel_size} {pixel_unit}')

## Optional: View images in napari

Confirm that you loaded the correct data

In [ ]:
import napari

colormaps_default = ['blue', 'green', 'yellow', 'red', 'cyan', 'magenta']

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
for i, (channel_name, image) in enumerate(images.items()):
    viewer.add_image(image, colormap=colormaps_default[i], name=channel_name, blending='additive', scale=pixel_size)


# **Step 2:** Blob detection

First, we detect blobs at pixel resolution using a Laplacian-of-Gaussian detector.

The main options that you can set are the **expected size of the blobs** (z,y,x - vector, in physical units, e.g. micron) and **intensity thresholds**.

In [ ]:
from skimage.feature import blob_log
from calmutils.localization.util import full_width_at_quantile_to_sigma

# relative threshold (Spinning Disk: 0.1-0.2, LSFM: 0.4-0.5, OMX: 0.2 -> detects lot of bg but works)
threshold = 0.2

# alternatively, set threshold per channel in dict
threshold = {
    '488 CSU-W1' : 0.15,
    '561 CSU-W1' : 0.12,
    '640 CSU-W1' : 0.15,
}

# expected size (FWHM) of subdiffraction beads -> PSF
expected_size = np.array([0.5, 0.25, 0.25])

# OMX Nup153: a bit smaller than confocal
# expected_size = np.array([0.4, 0.2, 0.2])

# for Drosophila LSFM data (lower resolution)
# expected_size = np.array([4, 1.5, 1.5])


# if we have one threshold, transform to dict with same value for all channels
if np.isscalar(threshold):
    threshold = {channel_name: threshold for channel_names in images.keys()}

# sigma for expected size in pixel
sigma_expected = full_width_at_quantile_to_sigma(expected_size / pixel_size)

blobs = {}
for channel_name, img in images.items():

    # blob_log with single sigma at expected size
    blobs_i = blob_log(img, min_sigma=sigma_expected, num_sigma=1, threshold=None, threshold_rel=threshold[channel_name])

    print(f'number of detected blobs in {channel_name}: {len(blobs_i)}')

    blobs[channel_name] = blobs_i

In [ ]:
import warnings
from scipy.optimize import OptimizeWarning

from calmutils.localization import refine_point_lsq

blobs_refined = {}
for channel_name, blobs_i in blobs.items():

    blobs_refined_i = []
    for blob in blobs_i:

        # do Gaussian fit, ignore warnings about failed optimization -> we will skip those blobs
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', (OptimizeWarning, RuntimeWarning))
            pos_refined, fit = refine_point_lsq(images[channel_name], blob[:3])
        
        # skip if fit not possible or negative sigma
        if fit is None:
            continue
        fit, _ = fit
        if np.any(fit[-3:] < 0) or np.any(np.isnan(fit)):
            continue

        # make position, sigmas vector like blob_log
        blobs_refined_i.append(list(pos_refined) + list(fit[-3:]))

    blobs_refined_i = np.array(blobs_refined_i)
    blobs_refined[channel_name] = blobs_refined_i
    
    print(f'remaining blobs in {channel_name} after subpixel refinement: {len(blobs_refined_i)}')

## Optional: plot detections in z-projection

In [ ]:
from matplotlib.patches import Ellipse
from calmutils.localization.util import get_ellipse_params

gamma = 2

for channel_name in images:
    
    plt.figure()
    plt.imshow(images[channel_name].max(axis=0)**(1/gamma), cmap='gray')
    plt.title(channel_name)

    blobs_refined_i = blobs_refined[channel_name]

    for blob in blobs_refined_i:
        pos_yx = blob[2:0:-1]
        sig_yx = blob[-1:3:-1]
        a, b, angle = get_ellipse_params(np.diag(sig_yx)**2)
        ell = Ellipse(pos_yx, a, b, angle=angle, color='red', fill=None)
        plt.gca().add_artist(ell)

# **Step 3:** Transformation Estimation

In [ ]:
from calmutils.descriptors import descriptor_local_qr

descriptors = {}
for channel_name, blobs_i in blobs_refined.items():
    # compute descriptors
    # NOTE: needs to be in isotropic coordinates to work, therefore multiply with pixel size
    descriptor, _ = descriptor_local_qr(blobs_i[:,:3] * pixel_size, progress_bar=False)
    descriptors[channel_name] = descriptor

In [ ]:
from skimage.feature import match_descriptors
from itertools import combinations

# how much closer a match has to be than the second best to be considered
descriptor_match_ratio = 5

matches = {}
for ch1, ch2 in combinations(images, 2):
    matches_i = match_descriptors(descriptors[ch1], descriptors[ch2], max_ratio=1/descriptor_match_ratio)
    matches[(ch1, ch2)] = matches_i
    print(f'{ch1} -> {ch2}: {len(matches_i)} matches')

## Optional: Plot matches

This plots two images with lines between matched points, **check that they mostly follow the same shift/transformation**

In [ ]:
from skimage.feature import plot_matches

fig, axs = plt.subplots(nrows=len(matches), figsize=(24, 8))
if len(matches) == 1:
    axs = [axs]
for ax, (ch1, ch2) in zip(axs, matches):
    # get x,y coordinates of keypoints
    keypoints1 = blobs_refined[ch1][:,1:3]
    keypoints2 = blobs_refined[ch2][:,1:3]

    # print(len(keypoints1))
    # print(len(keypoints2))

    img1_max = images[ch1].max(axis=0)
    img2_max = images[ch2].max(axis=0)

    plot_matches(ax, img1_max, img2_max, keypoints1, keypoints2, matches[(ch1, ch2)])
    ax.set_title(f'{ch1} -> {ch2}')

fig.tight_layout()

In [ ]:
from skimage.transform import AffineTransform, SimilarityTransform
from scipy.ndimage import affine_transform
from skimage.measure import ransac

# error threshold in RANSAC -> lower means more stringent filtering, but may lead to no transformation being estimated at all
residual_threshold = 0.5

# return AffineTransform constructor with specified dimensionality, would default to 2 otherwise
def affine_transform_nd(dimensionality):
    return lambda: AffineTransform(dimensionality=dimensionality)

# similarity: shift, rotate, scale
transform_type = SimilarityTransform

# alternative: full affine -> includes shearing, which may be undesired
# e.g., sometimes gave weird results on single layer of beads
# transform_type = affine_transform_nd(3)

transforms = {}
for (ch1, ch2), matches_i in matches.items():

    # get zyx of matched keypoints
    keypoints1 = blobs_refined[ch1][:,:3][matches_i.T[0]]
    keypoints2 = blobs_refined[ch2][:,:3][matches_i.T[1]]

    # pixel keypoints to physical unit -> we want transformation in that
    keypoints1 *= pixel_size
    keypoints2 *= pixel_size

    transform, inliers = ransac((keypoints1, keypoints2),
                        transform_type, 12, residual_threshold=residual_threshold, max_trials=5000)

    print(f'RANSAC on {ch1}->{ch2} inliers: {inliers.sum()}/{len(inliers)}')

    dist_before = (np.linalg.norm((keypoints1[inliers] - keypoints2[inliers]), axis=1).mean())
    dist_after = (np.linalg.norm((transform(keypoints1[inliers]) - keypoints2[inliers]), axis=1).mean())
    print(f'mean distance before transform: {dist_before:.3f} {pixel_unit}, after: {dist_after:.3f} {pixel_unit}')

    transforms[(ch1, ch2)] = transform.params
    transforms[(ch2, ch1)] = np.linalg.inv(transform.params)

# **Step 4:** Check alignment

Below, we align images of other channels to one reference channel and then display them in napari

In [ ]:
reference_channel = '561 CSU-W1'

images_aligned = {}
for ch, image in images.items():

    if ch == reference_channel:
        # identity transform
        mat = np.eye(image.ndim + 1)
    else:
        # NOTE: we want the inverse transform from ch to reference, i.e. the transform reference -> ch
        mat = transforms[(reference_channel, ch)]

    pixel_scale_mat = np.diag(list(pixel_size) + [1])
    mat = np.linalg.inv(pixel_scale_mat) @ mat @ pixel_scale_mat

    image_transformed = affine_transform(image, mat, order=2)
    images_aligned[ch] = image_transformed

    print(f'aligned image of channel {ch}')

View aligned results in napari -> channels should now overlap nicely

In [ ]:
import napari

colormaps_default = ['blue', 'green', 'yellow', 'red', 'cyan', 'magenta']

if napari.current_viewer() is not None:
    napari.current_viewer().close()

viewer = napari.Viewer()
for i, (channel_name, image) in enumerate(images_aligned.items()):
    viewer.add_image(image, colormap=colormaps_default[i], name=channel_name, scale=pixel_size, blending='additive')


# **Step 5**: Save transformations

In [ ]:
import json
from pathlib import Path

out_file = Path(image_path).parent / (Path(image_path).stem + '_channel_registration.json')

output = {
    'channels' : list(images.keys()),
    'pixel_size' : list(pixel_size),
    'size_unit' : pixel_unit,
    'z_direction' : z_direction,
    'field_of_view' : list(np.array(next(iter(images.values())).shape) * pixel_size),
    'source_file': image_path,
    'transforms' : [ {'channels' : k, 'parameters': list(v.flat)} for k,v in transforms.items()]
}

with open(out_file, 'w') as fd:
    json.dump(output, fd, indent=1)